### Importing the Libraries and loading data

In [ ]:
import pandas as pd 
import numpy as np
import warnings
from sklearn.model_selection import train_test_split


train_df = pd.read_csv('../datasets/cleaned datasets/train.csv')
test_df = pd.read_csv('../datasets/cleaned datasets/test.csv')

Lets first define our columns to encode and scale.

In [27]:
train_df.columns.tolist()

['id',
 'Podcast_Name',
 'episode_number',
 'Episode_Length_minutes',
 'Genre',
 'Host_Popularity_percentage',
 'Publication_Day',
 'Publication_Time',
 'Guest_Popularity_percentage',
 'Number_of_Ads',
 'Episode_Sentiment',
 'Listening_Time_minutes',
 'peak_time_to_publish',
 'is_night_or_evening',
 'is_weekend',
 'peak_popularity',
 'is_addless',
 'is_midweek',
 'is_monday',
 'total_popularity',
 'popularity_difference',
 'ad_density']

In [28]:
categorical_cols = ['Podcast_Name', 'Genre', 'Publication_Day', 'Publication_Time', 'Episode_Sentiment']
numeric_cols = ['episode_number', 'Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Number_of_Ads', 'total_popularity', 'popularity_difference', 'ad_density']

we should also define our input and target columns.

In [29]:
input_cols = [col for col in train_df.columns if col not in ['Listening_Time_minutes', 'id']]
target_cols = ['Listening_Time_minutes']

### Train/Validation Split

In [ ]:
train_df, val_df = train_test_split(train_df, test_size=0.25, random_state=42)

### Input/Target DFs

In [ ]:
train_inputs = train_df[input_cols]
train_targets = train_df[target_cols]

val_inputs = val_df[input_cols]
val_targets = val_df[target_cols]

test_inputs = test_df[input_cols]
# we dont have test targets as thats what we are trying to predict

### OneHotEncoding our categorical columns

In [ ]:
encoder = OneHotEncoder(sparse_output=False)
encoder.fit(train_inputs[categorical_cols])

warnings.filterwarnings('ignore')
encoded_cols = encoder.get_feature_names_out(categorical_cols)


train_inputs[encoded_cols] = encoder.transform(train_inputs[categorical_cols])
val_inputs[encoded_cols] = encoder.transform(val_inputs[categorical_cols])
test_inputs[encoded_cols] = encoder.transform(test_inputs[categorical_cols])

after the encoding, we can remove the columns we used to encode the features with. lets create a function for that

In [ ]:
def remove_encoded_cols(df):
    df = df.drop(columns=categorical_cols)
    return df    

train_inputs = remove_encoded_cols(train_inputs)
val_inputs = remove_encoded_cols(val_inputs)
test_inputs = remove_encoded_cols(test_inputs)

### Scaling Numerical Features (we will skip and try to see if scaling disturbs our model)

In [39]:
columns_to_scale = numeric_cols

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(train_inputs[columns_to_scale])

train_inputs[columns_to_scale] = scaler.transform(train_inputs[columns_to_scale])
val_inputs[columns_to_scale] = scaler.transform(val_inputs[columns_to_scale])
test_inputs[columns_to_scale] = scaler.transform(test_inputs[columns_to_scale])

In [42]:
train_inputs

,episode_number,Episode_Length_minutes,Host_Popularity_percentage,Guest_Popularity_percentage,Number_of_Ads,peak_time_to_publish,is_night_or_evening,is_weekend,peak_popularity,is_addless,...,Publication_Day_Thursday,Publication_Day_Tuesday,Publication_Day_Wednesday,Publication_Time_Afternoon,Publication_Time_Evening,Publication_Time_Morning,Publication_Time_Night,Episode_Sentiment_Negative,Episode_Sentiment_Neutral,Episode_Sentiment_Positive
0,0.979798,0.198330,0.744782,0.0000,0.000000,0,1,0,0,1,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,0.252525,0.368343,0.665147,0.7595,0.666667,0,0,1,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.151515,0.227217,0.695745,0.0897,0.000000,0,1,0,0,1,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0.444444,0.206524,0.566565,0.7870,0.666667,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,0.858586,0.339780,0.798075,0.5868,1.000000,0,0,0,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
749995,0.242424,0.232628,0.689564,0.0000,0.000000,0,0,1,0,1,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
749996,0.202020,0.232905,0.343566,0.0000,0.666667,1,1,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
749997,0.505051,0.095253,0.782979,0.8489,0.000000,0,0,0,0,1,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
749998,0.464646,0.335076,0.446707,0.9327,0.000000,0,0,0,0,1,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


### Rearranging the dataframes

In [ ]:
rest_of_the_cols_train = [col for col in train_inputs.columns if col not in numeric_cols]
rest_of_the_cols_vals = [col for col in val_inputs.columns if col not in numeric_cols]
rest_of_the_cols_test = [col for col in test_inputs.columns if col not in numeric_cols]

train_inputs = train_inputs[numeric_cols + rest_of_the_cols_train].copy()
val_inputs = val_inputs[numeric_cols + rest_of_the_cols_vals].copy()
test_inputs = test_inputs[numeric_cols + rest_of_the_cols_test].copy()

## Lets dump these dataframes to .csv file

In [ ]:
train_inputs.to_csv('../datasets/cleaned datasets/train_inputs.csv', index=None)
train_targets.to_csv('../datasets/cleaned datasets/train_targets.csv', index=None)

val_inputs.to_csv('../datasets/cleaned datasets/val_inputs.csv', index=None)
val_targets.to_csv('../datasets/cleaned datasets/val_targets.csv', index=None)

test_inputs.to_csv('../datasets/cleaned datasets/test_inputs.csv', index=None)